# 🚀 [0.94651] Pushing Beyond 0.94650 via 6-Model Meta-Stacking Lasso & Boundary Rules

### 📌 What This Notebook Does

The current top leaderboard plateau at **0.94650** is heavily dominated by public blends relying on `@jazivxt`'s baseline (`submission_latest_best.csv`) combined with `@taeyangg4`'s 4 deterministic boundary rules and secondary sorting. However, standard public blends leave 16,846 tied rank predictions unhandled.

This notebook pushes the LB score further to **0.94651** by:
1. **Rank Blending (5% Weight):** Blending $5\%$ rank-weight of our non-tied Lasso meta-predictions (lasso_test.csv) generated from my [S6E9 | 6-Model Leak-Free Meta-Stacking Engine](https://www.kaggle.com/code/sometimessubodh/s6e9-6-model-meta-stacking-engine) into the anchor.
2. **Applying 4 Synthetic Boundary Rules:** Applying target-purity shifts ($\pm 10, \pm 5$) on key deterministic feature rules (`Annual_Income_USD`, `Daily_Commute_km`, etc.).
3. **Lexicographical Tie-Breaking (lexsort):** Using our continuous Lasso model predictions from the [Meta-Stacking Engine](https://www.kaggle.com/code/sometimessubodh/s6e9-6-model-meta-stacking-engine) as a secondary sorter to break all 16,846 remaining tied rows.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import rankdata, spearmanr

In [ ]:
def rk(v):
    return rankdata(v, method="average") / len(v)

def lexrank(primary, secondary):
    N = len(primary)
    o = np.lexsort((secondary, primary))
    r = np.empty(N)
    r[o] = np.arange(1, N + 1)
    return (r - 0.5) / N

In [ ]:
# 1. Load Submissions & Test Features
base_sub = pd.read_csv("/kaggle/input/datasets/jazivxt/s6e9-zoom-zoom-baseline/submission_latest_best.csv").sort_values("id")
lasso_test = pd.read_csv("/kaggle/input/notebooks/sometimessubodh/s6e9-6-model-meta-stacking-engine/lasso_test_predictions.csv").sort_values("id")
test_df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/test.csv").sort_values("id").reset_index(drop=True)

ids = test_df["id"].to_numpy()
base_preds = base_sub["Will_Buy_EV"].to_numpy(dtype=float)
lasso_preds = lasso_test["lasso_test"].to_numpy(dtype=float)

In [ ]:
# 2. Extract Feature Masks for 4 Boundary Rules
inc = pd.to_numeric(test_df.Annual_Income_USD, errors="coerce").to_numpy()
km = pd.to_numeric(test_df.Daily_Commute_km, errors="coerce").to_numpy()
env1 = pd.to_numeric(test_df.Environmental_Concern_Level, errors="coerce").to_numpy() == 1
no_sub = test_df.Subsidy_Available.astype(str).to_numpy() == "No"
anx_mh = test_df.Range_Anxiety_Level.isin(["Medium", "High"]).to_numpy()

In [ ]:
# 3. Calculate Shift Adjustments
shift = np.zeros(len(test_df))
shift += np.where(inc >= 170537, +10.0, 0.0)                                    # High income (+10)
shift += np.where((inc >= 31004) & (inc <= 41970), -10.0, 0.0)                 # Dead-income zone (-10)
shift += np.where(km >= 83, -5.0, 0.0)                                          # High commute (-5)
shift += np.where((inc == 30000) & no_sub & (env1 | anx_mh), -5.0, 0.0)          # Low income / no subsidy (-5)

In [ ]:
# 4. Blend 5% Lasso + Post-Processing + Tie-breaking
# Rank blend primary base with 5% lasso
primary_blend = rk(base_preds) + 0.05 * rk(lasso_preds) + shift

# Tie-break using lasso_preds as the secondary sorter
final_ranks = lexrank(primary_blend, lasso_preds)

In [ ]:
# 5. Output Submission
sub = pd.DataFrame({"id": ids, "Will_Buy_EV": final_ranks})
sub.to_csv("submission_blend.csv", index=False)

print(f"Spearman against base anchor: {spearmanr(final_ranks, base_preds).statistic:.6f}")
print("Saved submission_blend.csv successfully.")

### 🙏 Acknowledgements & Credits

This notebook builds upon the brilliant insights and baseline work shared by the Kaggle community:

* **@jazivxt** :- For the core anchor predictions (`submission_latest_best.csv`, LB 0.94649) in his dataset [S6E9 Zoom Zoom Baseline](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline)
* **@taeyangg4** :- For discovering the 4 deterministic boundary rules on the synthetic training set and pioneering `lexsort` tie-breaking in his notebook [S6E9 094649 Multi Paradigm Lexsort Master
](https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master).
* **@yekenot** :- For the original [RealMLP PyTorch](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch)  baseline predictions used across early top blends.

*If this notebook helps your score or gives you useful ideas, an upvote is greatly appreciated!* 📈